# Interpolação e Extrapolação Populacional — Método Geométrico
### Todos os Municípios de MG (1970-2025), por Faixa Etária


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# utilitarios_comuns.py mora uma pasta acima deste notebook
# (Seleção e Transformação/População/, não .../População/Interpolação e Extrapolação/).
sys.path.append(str(Path.cwd().parent))

from utilitarios_comuns import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    POPULACAO_MINIMA_VALIDA,
    SUFIXO_COLUNA_TOTAL,
    PASTA_POPULACAO,
    CAMINHO_POPULACAO,
    FAIXAS_ETARIAS_PADRAO,
)

### Configurações

In [2]:
CAMINHO_ARQUIVO_ENTRADA = PASTA_POPULACAO / "tabela_populacao_faixa_etaria.csv"
CAMINHO_ARQUIVO_SAIDA = CAMINHO_POPULACAO

# Mesma lista de 8 faixas etárias usada em utilitarios_comuns.py (compartilhada
# com calculo_taxa.ipynb) -- reaproveitada em vez de redefinida aqui, para as
# duas não poderem divergir silenciosamente.
LISTA_FAIXAS_ETARIAS = list(FAIXAS_ETARIAS_PADRAO)

LISTA_ANOS_CENSITARIOS = [1970, 1980, 1991, 2000, 2010, 2022]
ANO_INICIO = 1970
ANO_FIM = 2025
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIO, ANO_FIM + 1))

### Leitura e Validação das Colunas

Confere se todas as colunas esperadas existem no arquivo de entrada:
uma coluna `<ano>_<faixa>` para cada combinação de ano censitário e
faixa etária, e uma coluna `<ano>_TOTAL` para cada ano censitário
(usada só como checagem/diagnóstico, não como fonte da interpolação).

In [3]:
tabela_populacao = pd.read_csv(CAMINHO_ARQUIVO_ENTRADA)

colunas_faixa_esperadas = []
for ano in LISTA_ANOS_CENSITARIOS:
    for faixa in LISTA_FAIXAS_ETARIAS:
        colunas_faixa_esperadas.append(f"{ano}_{faixa}")

colunas_total_esperadas = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_CENSITARIOS]

colunas_faltantes = [c for c in colunas_faixa_esperadas + colunas_total_esperadas if c not in tabela_populacao.columns]
if colunas_faltantes:
    raise ValueError(f"Colunas esperadas e não encontradas no arquivo de entrada: {colunas_faltantes}")

print(f"Arquivo de entrada validado: {tabela_populacao.shape[0]} municípios.")
print(f"Colunas de faixa etária esperadas ({len(colunas_faixa_esperadas)}): OK")
print(f"Colunas de total esperadas ({len(colunas_total_esperadas)}): OK")

tabela_populacao.head()

Arquivo de entrada validado: 853 municípios.
Colunas de faixa etária esperadas (48): OK
Colunas de total esperadas (6): OK


,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2022_40-49,2022_50-59,2022_60-69,2022_70 OU MAIS,1970_TOTAL,1980_TOTAL,1991_TOTAL,2000_TOTAL,2010_TOTAL,2022_TOTAL
0,abadia dos dourados,3100104,uberlandia,1542.0,1255.0,791.0,537.0,274.0,207.0,143.0,...,493,465,377,339,4848.0,3936.0,3252,3105,3676,3111
1,abaete,3100203,sete lagoas,3054.0,2786.0,1423.0,1115.0,874.0,470.0,311.0,...,1543,1752,1436,1250,10206.0,9926.0,10560,10947,11171,11474
2,abre campo,3100302,manhumirim,3015.0,2534.0,1462.0,1049.0,678.0,543.0,311.0,...,960,903,748,706,9779.0,9375.0,9131,6610,7238,6903
3,acaiaca,3100401,ponte nova,514.0,387.0,200.0,208.0,135.0,128.0,70.0,...,304,290,222,235,1699.0,1822.0,1937,1875,1881,2012
4,aguanil,3100807,divinopolis,359.0,302.0,217.0,187.0,126.0,75.0,23.0,...,298,303,260,215,1312.0,1560.0,1576,1753,1938,2196


### Preparar a tabela indexada por Código IBGE


In [4]:
tabela_indexada = tabela_populacao.set_index(COLUNA_CODIGO_IBGE, drop=False)

tabela_indexada[colunas_faixa_esperadas] = tabela_indexada[colunas_faixa_esperadas].where(
    tabela_indexada[colunas_faixa_esperadas] >= POPULACAO_MINIMA_VALIDA
)

tabela_indexada.head()

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2022_40-49,2022_50-59,2022_60-69,2022_70 OU MAIS,1970_TOTAL,1980_TOTAL,1991_TOTAL,2000_TOTAL,2010_TOTAL,2022_TOTAL
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,abadia dos dourados,3100104,uberlandia,1542.0,1255.0,791.0,537.0,274.0,207.0,143.0,...,493,465,377,339,4848.0,3936.0,3252,3105,3676,3111
3100203,abaete,3100203,sete lagoas,3054.0,2786.0,1423.0,1115.0,874.0,470.0,311.0,...,1543,1752,1436,1250,10206.0,9926.0,10560,10947,11171,11474
3100302,abre campo,3100302,manhumirim,3015.0,2534.0,1462.0,1049.0,678.0,543.0,311.0,...,960,903,748,706,9779.0,9375.0,9131,6610,7238,6903
3100401,acaiaca,3100401,ponte nova,514.0,387.0,200.0,208.0,135.0,128.0,70.0,...,304,290,222,235,1699.0,1822.0,1937,1875,1881,2012
3100807,aguanil,3100807,divinopolis,359.0,302.0,217.0,187.0,126.0,75.0,23.0,...,298,303,260,215,1312.0,1560.0,1576,1753,1938,2196


### Checagem de Valores Não Positivos, por Ano Censitário e Faixa Etária

Mostra, para cada combinação de ano e faixa, quantos municípios ficaram
sem dado válido -- útil para confirmar se o padrão bate com o esperado
(municípios ainda não emancipados nos censos mais antigos).

In [5]:
tabela_contagem_ausentes = pd.DataFrame(index=LISTA_ANOS_CENSITARIOS, columns=LISTA_FAIXAS_ETARIAS, dtype=int)

for ano in LISTA_ANOS_CENSITARIOS:
    for faixa in LISTA_FAIXAS_ETARIAS:
        coluna = f"{ano}_{faixa}"
        tabela_contagem_ausentes.loc[ano, faixa] = int(tabela_indexada[coluna].isna().sum())

print("Municípios sem dado válido (ausente/<= 0), por ano censitário x faixa etária:")
tabela_contagem_ausentes

Municípios sem dado válido (ausente/<= 0), por ano censitário x faixa etária:


,0-9,10-19,20-29,30-39,40-49,50-59,60-69,70 OU MAIS
1970,131.0,131.0,131.0,131.0,131.0,131.0,131.0,133.0
1980,131.0,131.0,131.0,130.0,131.0,131.0,131.0,132.0
1991,130.0,130.0,130.0,0.0,130.0,130.0,130.0,130.0
2000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2010,0.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Método Geométrico

Funções com as fórmulas do método geométrico discreto — as mesmas usadas
antes, agora aplicadas separadamente a cada faixa etária.

In [6]:
def calcular_taxa_geometrica(populacao_inicial, populacao_final, tempo_total):
    populacao_inicial_valida = populacao_inicial.where(populacao_inicial >= POPULACAO_MINIMA_VALIDA)
    razao = populacao_final / populacao_inicial_valida
    return razao ** (1 / tempo_total) - 1


def estimar_populacao_geometrica(populacao_base, taxa_geometrica, tempo_decorrido):
    return populacao_base * (1 + taxa_geometrica) ** tempo_decorrido

### Interpolação e Extrapolação

In [7]:
def interpolar_geometrica(tabela, colunas_censitarias, anos_censitarios, anos_completos):
    P = tabela[colunas_censitarias].copy()
    P.columns = anos_censitarios

    resultado = pd.DataFrame(index=P.index, columns=anos_completos, dtype=float)

    for ano in anos_censitarios:
        resultado[ano] = P[ano]

    for i in range(len(anos_censitarios) - 1):
        ano_inicial = anos_censitarios[i]
        ano_final = anos_censitarios[i + 1]

        populacao_inicial = P[ano_inicial]
        populacao_final = P[ano_final]
        tempo_total = ano_final - ano_inicial

        r_g = calcular_taxa_geometrica(populacao_inicial, populacao_final, tempo_total)

        anos_intermediarios = [ano for ano in anos_completos if ano_inicial < ano < ano_final]
        for ano in anos_intermediarios:
            tempo_decorrido = ano - ano_inicial
            resultado[ano] = estimar_populacao_geometrica(populacao_inicial, r_g, tempo_decorrido)

    return resultado


def extrapolar_geometrica(tabela_interpolada, ano_base_taxa, ano_base_extrapolacao, anos_a_extrapolar):
    '''
    ano_base_taxa/ano_base_extrapolacao: os dois últimos censos (2010, 2022),
    usados só para ESTIMAR r_g -- mesmo vocabulário usado nas células de
    visualização da taxa geométrica, para não ter dois nomes para o mesmo
    par de anos dentro do notebook. Quem é de fato extrapolado é
    ano_base_extrapolacao -> anos_a_extrapolar.
    '''
    P_base_taxa = tabela_interpolada[ano_base_taxa]
    P_base_extrapolacao = tabela_interpolada[ano_base_extrapolacao]
    t_taxa = ano_base_extrapolacao - ano_base_taxa

    r_g = calcular_taxa_geometrica(P_base_taxa, P_base_extrapolacao, t_taxa)

    extrapolacao = pd.DataFrame(index=tabela_interpolada.index)
    for ano in anos_a_extrapolar:
        extrapolacao[ano] = estimar_populacao_geometrica(P_base_extrapolacao, r_g, ano - ano_base_extrapolacao)

    return extrapolacao, r_g


ULTIMOS_DOIS_CENSOS = LISTA_ANOS_CENSITARIOS[-2:]  # [2010, 2022]
ultimo_ano_censitario = LISTA_ANOS_CENSITARIOS[-1]

ANOS_A_EXTRAPOLAR = [ano for ano in LISTA_ANOS_COMPLETOS if ano > ultimo_ano_censitario]

print("Últimos dois censos usados na taxa de extrapolação:", ULTIMOS_DOIS_CENSOS)
print("Anos extrapolados                                 :", ANOS_A_EXTRAPOLAR)

Últimos dois censos usados na taxa de extrapolação: [2010, 2022]
Anos extrapolados                                 : [2023, 2024, 2025]


### Interpolar e extrapolar cada faixa etária, separadamente

Para cada faixa, roda a mesma interpolação/extrapolação já usada antes
para o total — só que agora aplicada às colunas daquela faixa
especificamente. O resultado é um dicionário com uma tabela completa
(1970-2025) por faixa etária.

In [8]:
tabelas_por_faixa = {}

for faixa in LISTA_FAIXAS_ETARIAS:
    colunas_censitarias_da_faixa = [f"{ano}_{faixa}" for ano in LISTA_ANOS_CENSITARIOS]

    tabela_interpolada = interpolar_geometrica(
        tabela_indexada, colunas_censitarias_da_faixa, LISTA_ANOS_CENSITARIOS, LISTA_ANOS_COMPLETOS
    )

    tabela_extrapolada, _ = extrapolar_geometrica(
        tabela_interpolada,
        ano_base_taxa=ULTIMOS_DOIS_CENSOS[0],
        ano_base_extrapolacao=ULTIMOS_DOIS_CENSOS[1],
        anos_a_extrapolar=ANOS_A_EXTRAPOLAR,
    )

    tabela_completa_da_faixa = tabela_interpolada.copy()
    for ano in tabela_extrapolada.columns:
        tabela_completa_da_faixa[ano] = tabela_completa_da_faixa[ano].fillna(tabela_extrapolada[ano])

    # Arredonda para inteiro (contagem de pessoas), sempre que possível,
    # antes de somar as faixas -- evita acumular erro de arredondamento
    # entre as 8 somas parciais. Mantido como float64 (e não Int64): o
    # restante do pipeline (taxas) já é float por causa de NaN e de operações
    # subsequentes -- um único tipo numérico em todo o schema, em vez de
    # Int64 só nos CSVs de população.
    tabela_completa_da_faixa = (
        np.round(tabela_completa_da_faixa)
        .replace([np.inf, -np.inf], np.nan)
        .astype("float64")
    )

    tabelas_por_faixa[faixa] = tabela_completa_da_faixa

print(f"Faixas etárias processadas: {list(tabelas_por_faixa.keys())}")
tabelas_por_faixa[LISTA_FAIXAS_ETARIAS[0]].head()

Faixas etárias processadas: ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70 OU MAIS']


,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,1542.0,1458.0,1379.0,1304.0,1233.0,1166.0,1103.0,1043.0,986.0,933.0,...,360.0,355.0,349.0,343.0,338.0,332.0,327.0,322.0,317.0,312.0
3100203,3054.0,2920.0,2792.0,2670.0,2553.0,2442.0,2335.0,2233.0,2135.0,2041.0,...,1288.0,1267.0,1247.0,1227.0,1207.0,1188.0,1169.0,1150.0,1132.0,1114.0
3100302,3015.0,2939.0,2864.0,2791.0,2721.0,2652.0,2584.0,2519.0,2455.0,2393.0,...,869.0,854.0,840.0,826.0,813.0,799.0,786.0,773.0,760.0,748.0
3100401,514.0,510.0,505.0,501.0,496.0,492.0,488.0,484.0,479.0,475.0,...,233.0,228.0,223.0,219.0,215.0,210.0,206.0,202.0,198.0,194.0
3100807,359.0,360.0,362.0,363.0,365.0,366.0,367.0,369.0,370.0,372.0,...,264.0,265.0,265.0,266.0,267.0,267.0,268.0,269.0,269.0,270.0


### População Interpolada/Extrapolada por Faixa Etária (Total do Estado)

`tabelas_por_faixa` guarda uma tabela por faixa etária, com uma linha
por município -- grande demais para olhar direto. Para ver o resultado
da interpolação/extrapolação de cada faixa etária ano a ano (e não só
a soma de todas as faixas, que é o que `tabela_populacao_total` mostra
mais abaixo), soma-se cada `tabelas_por_faixa[faixa]` por todos os
municípios, com uma linha por ano (1970-2025) e uma coluna por faixa
etária.

In [9]:
tabela_populacao_total_por_faixa = pd.DataFrame(
    {faixa: tabelas_por_faixa[faixa].sum() for faixa in LISTA_FAIXAS_ETARIAS},
    index=LISTA_ANOS_COMPLETOS,
)
tabela_populacao_total_por_faixa.index.name = "Ano"

tabela_populacao_total_por_faixa_formatada = (
    tabela_populacao_total_por_faixa.style
    .format("{:,.0f}")
    .set_caption("População interpolada/extrapolada por faixa etária, por ano (total do estado)")
)

tabela_populacao_total_por_faixa_formatada

,0-9,10-19,20-29,30-39,40-49,50-59,60-69,70 OU MAIS
Ano,,,,,,,,
1970,"1,686,715","1,459,944","911,235","650,135","467,737","314,208","180,088","102,544"
1971,"1,681,349","1,471,118","931,157","696,351","477,262","321,621","186,558","107,134"
1972,"1,677,330","1,483,308","952,552","747,067","487,252","329,414","193,392","112,066"
1973,"1,674,641","1,496,648","975,483","802,856","497,726","337,531","200,564","117,296"
1974,"1,673,391","1,511,138","1,000,102","864,319","508,749","346,041","208,110","122,854"
1975,"1,673,532","1,526,815","1,026,445","932,139","520,253","354,922","216,058","128,740"
1976,"1,675,174","1,543,835","1,054,726","1,007,242","532,442","364,273","224,471","135,051"
1977,"1,678,365","1,562,193","1,085,066","1,090,524","545,195","374,043","233,300","141,741"
1978,"1,683,153","1,582,003","1,117,572","1,183,144","558,665","384,291","242,615","148,872"


### Somar as faixas etárias para obter a população total de cada ano

A população total do ano deixa de ser interpolada diretamente -- passa
a ser a soma das 8 faixas já interpoladas/extrapoladas, para cada
município e cada ano.

In [10]:
tabela_populacao_total = pd.DataFrame(index=tabela_indexada.index, columns=LISTA_ANOS_COMPLETOS)

for ano in LISTA_ANOS_COMPLETOS:
    soma_das_faixas = None
    for faixa in LISTA_FAIXAS_ETARIAS:
        valor_da_faixa = tabelas_por_faixa[faixa][ano]
        soma_das_faixas = valor_da_faixa if soma_das_faixas is None else soma_das_faixas.add(valor_da_faixa, fill_value=0)
    tabela_populacao_total[ano] = soma_das_faixas

tabela_populacao_total = tabela_populacao_total.astype("float64")

novos_nomes_das_colunas = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_COMPLETOS]
tabela_populacao_total.columns = novos_nomes_das_colunas

tabela_populacao_total.head()

,1970_TOTAL,1971_TOTAL,1972_TOTAL,1973_TOTAL,1974_TOTAL,1975_TOTAL,1976_TOTAL,1977_TOTAL,1978_TOTAL,1979_TOTAL,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848.0,4735.0,4624.0,4523.0,4424.0,4332.0,4242.0,4159.0,4083.0,4007.0,...,3277.0,3233.0,3195.0,3164.0,3142.0,3123.0,3111.0,3105.0,3106.0,3115.0
3100203,10206.0,10128.0,10062.0,10007.0,9961.0,9929.0,9906.0,9894.0,9894.0,9903.0,...,11149.0,11177.0,11215.0,11263.0,11322.0,11392.0,11474.0,11569.0,11677.0,11799.0
3100302,9779.0,9720.0,9664.0,9613.0,9568.0,9524.0,9484.0,9451.0,9420.0,9397.0,...,6956.0,6933.0,6914.0,6901.0,6895.0,6895.0,6903.0,6917.0,6936.0,6963.0
3100401,1699.0,1710.0,1719.0,1730.0,1742.0,1753.0,1767.0,1780.0,1793.0,1806.0,...,1918.0,1930.0,1942.0,1955.0,1974.0,1992.0,2012.0,2035.0,2057.0,2084.0
3100807,1312.0,1332.0,1353.0,1374.0,1396.0,1420.0,1446.0,1473.0,1500.0,1528.0,...,2042.0,2065.0,2087.0,2112.0,2139.0,2167.0,2196.0,2228.0,2261.0,2297.0


### Total recalculado bate com o total original nos anos censitários?



In [11]:
divergencias_encontradas = 0

for ano in LISTA_ANOS_CENSITARIOS:
    coluna_total_original = f"{ano}{SUFIXO_COLUNA_TOTAL}"
    coluna_total_recalculada = f"{ano}{SUFIXO_COLUNA_TOTAL}"

    total_original = tabela_indexada[coluna_total_original]
    total_recalculado = tabela_populacao_total[coluna_total_recalculada]

    diferenca_absoluta = (total_original - total_recalculado).abs()
    quantidade_com_diferenca_relevante = int((diferenca_absoluta > 1).sum())  # tolerância de 1 hab. por arredondamento

    if quantidade_com_diferenca_relevante > 0:
        divergencias_encontradas += quantidade_com_diferenca_relevante
        print(
            f"ATENÇÃO ({ano}): {quantidade_com_diferenca_relevante} município(s) com diferença "
            f"maior que 1 habitante entre o total original e a soma das faixas."
        )

if divergencias_encontradas == 0:
    print("Checagem OK: em todos os anos censitários, a soma das faixas bate com o total original (± 1 hab.).")

Checagem OK: em todos os anos censitários, a soma das faixas bate com o total original (± 1 hab.).


### Visualização das Taxas Geométricas Calculadas

A taxa geométrica $r_g$ é calculada uma vez por intervalo entre censos
consecutivos (interpolação) e mais uma vez entre os dois últimos censos
(extrapolação), separadamente para cada faixa etária. Para consolidar
cada cálculo num único número (em vez de um valor por município),
usa-se a população total do estado (soma de todos os municípios) em
cada faixa etária -- gerando uma tabela com $P_{inicial}$, $P_{final}$,
$t$, $r_g$ e $P(t)$ para cada cálculo, além do gráfico de $r_g$ ao
longo do tempo.

In [12]:
def formatar_intervalo_anos(anos):
    if not anos:
        return "--"
    if len(anos) == 1:
        return str(anos[0])
    return f"{min(anos)}–{max(anos)}"


registros_taxas_geometricas = []

for faixa in LISTA_FAIXAS_ETARIAS:
    colunas_censitarias_da_faixa = [f"{ano}_{faixa}" for ano in LISTA_ANOS_CENSITARIOS]
    populacao_estado_por_ano = tabela_indexada[colunas_censitarias_da_faixa].sum()
    populacao_estado_por_ano.index = LISTA_ANOS_CENSITARIOS

    # Um cálculo de r_g por intervalo entre censos consecutivos (interpolação)
    for i in range(len(LISTA_ANOS_CENSITARIOS) - 1):
        ano_inicial = LISTA_ANOS_CENSITARIOS[i]
        ano_final = LISTA_ANOS_CENSITARIOS[i + 1]

        P_inicial = populacao_estado_por_ano[ano_inicial]
        P_final = populacao_estado_por_ano[ano_final]
        t = ano_final - ano_inicial
        # Reusa calcular_taxa_geometrica (célula 7494ba1d) -- P_inicial/P_final
        # são escalares, então são embrulhados numa Series de 1 elemento só
        # para poder passar pela guarda de POPULACAO_MINIMA_VALIDA (.where),
        # que exige um objeto pandas.
        r_g = float(calcular_taxa_geometrica(pd.Series([P_inicial]), pd.Series([P_final]), t).iloc[0])

        # r_g é ajustado para bater exatamente com os dois censos (ano_inicial
        # e ano_final); só os anos ENTRE eles são de fato estimados pela fórmula.
        anos_estimados = [ano for ano in LISTA_ANOS_COMPLETOS if ano_inicial < ano < ano_final]

        registros_taxas_geometricas.append({
            "Faixa Etária": faixa,
            "Tipo de Cálculo": "Interpolação",
            "Ano Inicial": ano_inicial,
            "Ano Final": ano_final,
            "Anos Estimados": formatar_intervalo_anos(anos_estimados),
            "P_inicial": P_inicial,
            "P_final": P_final,
            "t": t,
            "Ano para Calcular Taxa": formatar_intervalo_anos([ano_inicial, ano_final]),
            "r_g": r_g,
        })

    # Um cálculo de r_g entre os dois últimos censos (usado na extrapolação) --
    # aqui "Ano Inicial"/"P_inicial" são o censo de 2022 (base da extrapolação),
    # não o de 2010: o par (2010, 2022) só serve para ESTIMAR r_g -- ver coluna
    # "Ano para Calcular Taxa" -- quem é efetivamente extrapolado é 2022 -> anos futuros.
    ano_base_taxa, ano_base_extrapolacao = ULTIMOS_DOIS_CENSOS
    P_base_taxa = populacao_estado_por_ano[ano_base_taxa]
    P_base_extrapolacao = populacao_estado_por_ano[ano_base_extrapolacao]
    t_taxa = ano_base_extrapolacao - ano_base_taxa
    r_g = float(calcular_taxa_geometrica(pd.Series([P_base_taxa]), pd.Series([P_base_extrapolacao]), t_taxa).iloc[0])

    ultimo_ano_extrapolado = ANOS_A_EXTRAPOLAR[-1]
    P_extrapolada = estimar_populacao_geometrica(P_base_extrapolacao, r_g, ultimo_ano_extrapolado - ano_base_extrapolacao)

    registros_taxas_geometricas.append({
        "Faixa Etária": faixa,
        "Tipo de Cálculo": "Extrapolação",
        "Ano Inicial": ano_base_extrapolacao,
        "Ano Final": ultimo_ano_extrapolado,
        "Anos Estimados": formatar_intervalo_anos(ANOS_A_EXTRAPOLAR),
        "P_inicial": P_base_extrapolacao,
        "P_final": P_extrapolada,
        "t": ultimo_ano_extrapolado - ano_base_extrapolacao,
        "Ano para Calcular Taxa": formatar_intervalo_anos([ano_base_taxa, ano_base_extrapolacao]),
        "r_g": r_g,
    })

tabela_taxas_geometricas = pd.DataFrame(registros_taxas_geometricas)
#tabela_taxas_geometricas

In [13]:
tabela_taxas_formatada = tabela_taxas_geometricas.style.format({
    "P_inicial": "{:,.0f}",
    "P_final": "{:,.0f}",
    "t": "{:.0f}",
    "r_g": "{:+.4%}",
}).set_caption(
    "Taxas geométricas por faixa etária e intervalo (população total do estado)"
).hide(axis="index")

tabela_taxas_formatada

Faixa Etária,Tipo de Cálculo,Ano Inicial,Ano Final,Anos Estimados,P_inicial,P_final,t,Ano para Calcular Taxa,r_g
0-9,Interpolação,1970,1980,1971–1979,"1,686,715","1,698,029",10,1970–1980,+0.0669%
0-9,Interpolação,1980,1991,1981–1990,"1,698,029","1,729,055",11,1980–1991,+0.1647%
0-9,Interpolação,1991,2000,1992–1999,"1,729,055","1,622,389",9,1991–2000,-0.7050%
0-9,Interpolação,2000,2010,2001–2009,"1,622,389","1,329,502",10,2000–2010,-1.9713%
0-9,Interpolação,2010,2022,2011–2021,"1,329,502","1,201,038",12,2010–2022,-0.8432%
0-9,Extrapolação,2022,2025,2023–2025,"1,201,038","1,170,910",3,2010–2022,-0.8432%
10-19,Interpolação,1970,1980,1971–1979,"1,459,944","1,626,338",10,1970–1980,+1.0852%
10-19,Interpolação,1980,1991,1981–1990,"1,626,338","1,698,975",11,1980–1991,+0.3980%
10-19,Interpolação,1991,2000,1992–1999,"1,698,975","1,800,993",9,1991–2000,+0.6500%
10-19,Interpolação,2000,2010,2001–2009,"1,800,993","1,680,533",10,2000–2010,-0.6899%


In [14]:
import plotly.graph_objects as go

fig = go.Figure()

for faixa in LISTA_FAIXAS_ETARIAS:
    dados_faixa = tabela_taxas_geometricas[
        tabela_taxas_geometricas["Faixa Etária"] == faixa
    ].sort_values("Ano Inicial")

    fig.add_trace(
        go.Scatter(
            x=dados_faixa["Ano Inicial"],
            y=dados_faixa["r_g"] * 100,
            mode="lines+markers",
            name=faixa,
            hovertemplate=(
                "Faixa etária: %{fullData.name}<br>"
                "Ano inicial: %{x}<br>"
                "Taxa geométrica: %{y:.2f}% ao ano"
                "<extra></extra>"
            ),
        )
    )

# Marca visualmente onde termina a interpolação
# e começa a extrapolação
fig.add_vline(
    x=ULTIMOS_DOIS_CENSOS[0],
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

# Linha horizontal em y = 0
fig.add_hline(
    y=0,
    line_color="gray",
    line_width=1,
)

fig.update_layout(
    title="Taxas geométricas calculadas por faixa etária e intervalo",
    xaxis_title="Ano inicial do intervalo",
    yaxis_title="Taxa geométrica r<sub>g</sub> (% ao ano)",
    legend_title="Faixa Etária",
    template="plotly_white",
    hovermode="x unified",
    width=1200,
    height=600,
)

fig.show()

### Versão Resumida — Taxa Geométrica da População Total

A tabela acima tem 48 linhas (8 faixas etárias × 6 cálculos) — grande
demais para uma tabela de texto. Esta versão resume para 6 linhas,
calculando $r_g$ diretamente sobre a população **total** do estado
(coluna `<ano>_TOTAL`, já validada na checagem acima) em vez de uma
linha por faixa etária. É a versão mais curta para colocar no
Overleaf; a tabela completa por faixa etária fica disponível acima
como referência/apêndice.

In [15]:
colunas_total_censitarias = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_CENSITARIOS]
populacao_total_estado_por_ano = tabela_indexada[colunas_total_censitarias].sum()
populacao_total_estado_por_ano.index = LISTA_ANOS_CENSITARIOS

registros_taxas_resumo = []

for i in range(len(LISTA_ANOS_CENSITARIOS) - 1):
    ano_inicial = LISTA_ANOS_CENSITARIOS[i]
    ano_final = LISTA_ANOS_CENSITARIOS[i + 1]

    P_inicial = populacao_total_estado_por_ano[ano_inicial]
    P_final = populacao_total_estado_por_ano[ano_final]
    t = ano_final - ano_inicial
    # Reusa calcular_taxa_geometrica (célula 7494ba1d) -- ver comentário
    # equivalente na célula anterior sobre o embrulho em Series de 1 elemento.
    r_g = float(calcular_taxa_geometrica(pd.Series([P_inicial]), pd.Series([P_final]), t).iloc[0])

    anos_estimados = [ano for ano in LISTA_ANOS_COMPLETOS if ano_inicial < ano < ano_final]

    registros_taxas_resumo.append({
        "Tipo de Cálculo": "Interpolação",
        "Ano Inicial": ano_inicial,
        "Ano Final": ano_final,
        "Anos Estimados": formatar_intervalo_anos(anos_estimados),
        "P_inicial": P_inicial,
        "P_final": P_final,
        "t": t,
        "Ano para Calcular Taxa": formatar_intervalo_anos([ano_inicial, ano_final]),
        "r_g": r_g,
    })

# "Ano Inicial"/"P_inicial" da extrapolação são o censo de 2022 (base da
# extrapolação); o par (2010, 2022) só serve para estimar r_g -- ver coluna
# "Ano para Calcular Taxa".
ano_base_taxa, ano_base_extrapolacao = ULTIMOS_DOIS_CENSOS
P_base_taxa = populacao_total_estado_por_ano[ano_base_taxa]
P_base_extrapolacao = populacao_total_estado_por_ano[ano_base_extrapolacao]
t_taxa = ano_base_extrapolacao - ano_base_taxa
r_g = float(calcular_taxa_geometrica(pd.Series([P_base_taxa]), pd.Series([P_base_extrapolacao]), t_taxa).iloc[0])

ultimo_ano_extrapolado = ANOS_A_EXTRAPOLAR[-1]
P_extrapolada = estimar_populacao_geometrica(P_base_extrapolacao, r_g, ultimo_ano_extrapolado - ano_base_extrapolacao)

registros_taxas_resumo.append({
    "Tipo de Cálculo": "Extrapolação",
    "Ano Inicial": ano_base_extrapolacao,
    "Ano Final": ultimo_ano_extrapolado,
    "Anos Estimados": formatar_intervalo_anos(ANOS_A_EXTRAPOLAR),
    "P_inicial": P_base_extrapolacao,
    "P_final": P_extrapolada,
    "t": ultimo_ano_extrapolado - ano_base_extrapolacao,
    "Ano para Calcular Taxa": formatar_intervalo_anos([ano_base_taxa, ano_base_extrapolacao]),
    "r_g": r_g,
})

tabela_taxas_resumo = pd.DataFrame(registros_taxas_resumo)

tabela_taxas_resumo_formatada = tabela_taxas_resumo.style.format({
    "P_inicial": "{:,.0f}",
    "P_final": "{:,.0f}",
    "t": "{:.0f}",
    "r_g": "{:+.4%}",
}).set_caption(
    "Taxa geométrica por intervalo censitário (população total do estado)"
).hide(axis="index")

tabela_taxas_resumo_formatada

Tipo de Cálculo,Ano Inicial,Ano Final,Anos Estimados,P_inicial,P_final,t,Ano para Calcular Taxa,r_g
Interpolação,1970,1980,1971–1979,"5,772,606","7,339,191",10,1970–1980,+2.4301%
Interpolação,1980,1991,1981–1990,"7,339,191","8,346,023",11,1980–1991,+1.1756%
Interpolação,1991,2000,1992–1999,"8,346,023","9,255,322",9,1991–2000,+1.1557%
Interpolação,2000,2010,2001–2009,"9,255,322","9,236,906",10,2000–2010,-0.0199%
Interpolação,2010,2022,2011–2021,"9,236,906","10,524,280",12,2010–2022,+1.0932%
Extrapolação,2022,2025,2023–2025,"10,524,280","10,873,237",3,2010–2022,+1.0932%


## Exportação

Combina o resultado em uma única tabela por município, com **todas as
8 faixas etárias interpoladas/extrapoladas para cada ano de 1970 a
2025**, mais a população total (soma das faixas) de cada ano,
preservando `REGIONAL DE SAÚDE` vinda do arquivo de entrada. O padrão
de nomeação das colunas segue o mesmo do arquivo de entrada: todas as
colunas `<ano>_<faixa>` primeiro, depois o bloco de `<ano>_TOTAL`.

In [16]:

blocos_por_faixa = {}
for ano in LISTA_ANOS_COMPLETOS:
    for faixa in LISTA_FAIXAS_ETARIAS:
        blocos_por_faixa[f"{ano}_{faixa}"] = tabelas_por_faixa[faixa][ano]

tabela_faixas_completa = pd.DataFrame(blocos_por_faixa, index=tabela_indexada.index).astype("float64")

colunas_faixa_ordenadas = [f"{ano}_{faixa}" for ano in LISTA_ANOS_COMPLETOS for faixa in LISTA_FAIXAS_ETARIAS]
colunas_total_ordenadas = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_COMPLETOS]

tabela_faixas_completa = tabela_faixas_completa[colunas_faixa_ordenadas]
tabela_populacao_total = tabela_populacao_total[colunas_total_ordenadas]

tabela_saida = tabela_indexada[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, COLUNA_REGIONAL]].merge(
    tabela_faixas_completa, left_index=True, right_index=True
).merge(
    tabela_populacao_total, left_index=True, right_index=True
)

CAMINHO_ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
tabela_saida.to_csv(CAMINHO_ARQUIVO_SAIDA, index=False)

print("Arquivo salvo em:", CAMINHO_ARQUIVO_SAIDA)
print("Shape:", tabela_saida.shape)
print(f"Período coberto: {ANO_INICIO}-{ANO_FIM} ({len(LISTA_ANOS_COMPLETOS)} anos)")
print(f"Colunas de faixa etária: {len(colunas_faixa_ordenadas)} | Colunas de total: {len(colunas_total_ordenadas)}")
tabela_saida.head()

Arquivo salvo em: /home/anna/tcc/Seleção e Transformação/População/Interpolação e Extrapolação/populacao_interpolacao_extrapolacao.csv
Shape: (853, 507)
Período coberto: 1970-2025 (56 anos)
Colunas de faixa etária: 448 | Colunas de total: 56


,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,abadia dos dourados,3100104,uberlandia,1542.0,1255.0,791.0,537.0,274.0,207.0,143.0,...,3277.0,3233.0,3195.0,3164.0,3142.0,3123.0,3111.0,3105.0,3106.0,3115.0
3100203,abaete,3100203,sete lagoas,3054.0,2786.0,1423.0,1115.0,874.0,470.0,311.0,...,11149.0,11177.0,11215.0,11263.0,11322.0,11392.0,11474.0,11569.0,11677.0,11799.0
3100302,abre campo,3100302,manhumirim,3015.0,2534.0,1462.0,1049.0,678.0,543.0,311.0,...,6956.0,6933.0,6914.0,6901.0,6895.0,6895.0,6903.0,6917.0,6936.0,6963.0
3100401,acaiaca,3100401,ponte nova,514.0,387.0,200.0,208.0,135.0,128.0,70.0,...,1918.0,1930.0,1942.0,1955.0,1974.0,1992.0,2012.0,2035.0,2057.0,2084.0
3100807,aguanil,3100807,divinopolis,359.0,302.0,217.0,187.0,126.0,75.0,23.0,...,2042.0,2065.0,2087.0,2112.0,2139.0,2167.0,2196.0,2228.0,2261.0,2297.0


### Exportação — Versão Agrupada por Regional de Saúde

Gera um segundo CSV a partir do que acabou de ser salvo, sem
sobrescrevê-lo: soma a população de todos os municípios de cada
`REGIONAL DE SAÚDE`, removendo as colunas `MUNICÍPIO` e
`CÓDIGO IBGE` (não fazem sentido depois do agrupamento). Mesma
estrutura de colunas do arquivo por município (`<ano>_<faixa>` e
`<ano>_TOTAL`), só que agora uma linha por regional em vez de uma
linha por município.

In [17]:
CAMINHO_ARQUIVO_SAIDA_REGIONAL = CAMINHO_ARQUIVO_SAIDA.with_name(
    CAMINHO_ARQUIVO_SAIDA.stem + "_por_regional.csv"
)

colunas_de_populacao = colunas_faixa_ordenadas + colunas_total_ordenadas

tabela_saida_por_regional = (
    tabela_saida.groupby(COLUNA_REGIONAL, as_index=False)[colunas_de_populacao]
    .sum(min_count=1)  # min_count=1: se todos os municípios da regional forem NaN, mantém NaN, não 0
)

tabela_saida_por_regional.to_csv(CAMINHO_ARQUIVO_SAIDA_REGIONAL, index=False)

print("Arquivo salvo em:", CAMINHO_ARQUIVO_SAIDA_REGIONAL)
print("Shape:", tabela_saida_por_regional.shape, f"({tabela_saida_por_regional[COLUNA_REGIONAL].nunique()} regionais)")
tabela_saida_por_regional.head()

Arquivo salvo em: /home/anna/tcc/Seleção e Transformação/População/Interpolação e Extrapolação/populacao_interpolacao_extrapolacao_por_regional.csv
Shape: (28, 505) (28 regionais)


,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,1970_70 OU MAIS,1971_0-9,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
0,alfenas,38783.0,35453.0,20531.0,15402.0,12313.0,8885.0,5314.0,2762.0,38693.0,...,220573.0,222548.0,224726.0,227098.0,229683.0,232490.0,235547.0,238845.0,242392.0,246231.0
1,barbacena,43565.0,38879.0,24384.0,17829.0,13842.0,9527.0,5567.0,2878.0,43476.0,...,242479.0,244759.0,247335.0,250227.0,253461.0,257039.0,261518.0,265379.0,270200.0,275488.0
2,belo horizonte,254997.0,224526.0,170568.0,118602.0,80361.0,54527.0,31008.0,17094.0,261563.0,...,2475206.0,2511888.0,2553195.0,2599714.0,2652117.0,2711218.0,2790345.0,2853706.0,2939740.0,3037868.0
3,coronel fabriciano,69445.0,56207.0,33644.0,24399.0,16147.0,10017.0,5421.0,2867.0,69684.0,...,372070.0,374307.0,377064.0,380355.0,384231.0,388702.0,406050.0,399537.0,405976.0,413156.0
4,diamantina,53164.0,43585.0,25834.0,18715.0,13531.0,9649.0,5950.0,3489.0,53019.0,...,195878.0,195433.0,195195.0,195164.0,195343.0,195745.0,198264.0,197187.0,198238.0,199528.0
